In [1]:
# import all required libraries
import pandas as pd
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import get_close_matches

In [2]:
# import both datasets
credits = pd.read_csv('tmdb_5000_credits.csv')
movies = pd.read_csv('tmdb_5000_movies.csv')

# **Data Understanding**

In [3]:
# top 5 rows of movies dataset
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [4]:
# top 5 rows of credits dataset
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [5]:
# shape of both datasets
print(f'movies shape {movies.shape}')
print(f'credits shape {credits.shape}')

movies shape (4803, 20)
credits shape (4803, 4)


Movies dataset have 20 columns and 4803 rows and Credits dataset have 4 columns and 4803 rows.

In [6]:
# datatype of columns in movies dataset
movies.dtypes

budget                    int64
genres                      str
homepage                    str
id                        int64
keywords                    str
original_language           str
original_title              str
overview                    str
popularity              float64
production_companies        str
production_countries        str
release_date                str
revenue                   int64
runtime                 float64
spoken_languages            str
status                      str
tagline                     str
title                       str
vote_average            float64
vote_count                int64
dtype: object

In [7]:
# datatype of columns in credits dataset
credits.dtypes

movie_id    int64
title         str
cast          str
crew          str
dtype: object

In [8]:
# number of null values in movies dataset columns
movies.isnull().sum().sort_values(ascending=False)

homepage                3091
tagline                  844
overview                   3
runtime                    2
release_date               1
id                         0
budget                     0
genres                     0
original_title             0
popularity                 0
original_language          0
keywords                   0
production_countries       0
production_companies       0
spoken_languages           0
revenue                    0
status                     0
title                      0
vote_average               0
vote_count                 0
dtype: int64

There are 3091 null values in homepage column, 844 in tagline, 3 in overview, 2 in runtime, 1 in release_date.

In [9]:
# number of null values in credits dataset columns
credits.isnull().sum()

movie_id    0
title       0
cast        0
crew        0
dtype: int64

There is no null value in credits dataset.

In [10]:
print(f'movies duplicates {movies.duplicated().sum()}')
print(f'credits duplicates {credits.duplicated().sum()}')

movies duplicates 0
credits duplicates 0


There is no duplicate row sample in both datasets.

# **Data Preprocessing**

In [11]:
# merge both datasets into one
movies = movies.merge(credits, on='title')

In [ ]:
# select required columns
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew','vote_average','vote_count']]

In [14]:
# drop null values
movies.dropna(inplace=True)

# **Data Transformation**

In [15]:
# extract name from genres and keywords columns
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [16]:
# extract top 3 cast from cast column
def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(convert_cast)

In [17]:
# extract director from crew column
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [18]:
# remove spaces between words
def collapse(L):
    return [i.replace(" ", "") for i in L]

movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)

In [19]:
# remove punctuations from overview column
def clean_overview(text):
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return text.split()

movies['overview'] = movies['overview'].apply(clean_overview)

# **Feature Engineering**

In [20]:
# create tags column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [21]:
# create new dataframe
new_df = movies[['movie_id','title','tags','vote_average','vote_count','popularity']]

new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

new_df['title'] = new_df['title'].apply(lambda x: x.lower())

# **Popularity-Based Recommendation**

In [22]:
# calculate weighted score
C = new_df['vote_average'].mean()
m = new_df['vote_count'].quantile(0.9)

def weighted_rating(x):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m))*R + (m/(v+m))*C

new_df['score'] = new_df.apply(weighted_rating, axis=1)

In [23]:
# top 10 popular movies
new_df.sort_values(by='score', ascending=False).head(10)['title']

1883                         the shawshank redemption
662                                        fight club
65                                    the dark knight
3235                                     pulp fiction
96                                          inception
3340                                    the godfather
95                                       interstellar
809                                      forrest gump
329     the lord of the rings: the return of the king
1992                          the empire strikes back
Name: title, dtype: str

# **Content-Based Recommendation System**

In [ ]:
# apply vectorization on tags column
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = tfidf.fit_transform(new_df['tags']).toarray()

In [34]:
# create dictonary to store movies index
movie_dict = {title: idx for idx, title in enumerate(new_df['title'])}

In [35]:
# recommend function
def recommend_movie(movie):

    movie = movie.lower()

    # ᐧ Case 1: Exact match (O(1))
    idx = movie_dict.get(movie)
    if idx is not None:
        sim_score = cosine_similarity(vectors[idx].reshape(1, -1), vectors).flatten()
        sim_indices = sim_score.argsort()[::-1][1:11]
        return [new_df['title'].iloc[i].title() for i in sim_indices]

    # ᐧ Case 2: Fuzzy match
    close_match = get_close_matches(movie, new_df['title'].tolist(), n=1, cutoff=0.6)
    if close_match:
        matched = close_match[0]
        return recommend_movie(matched)

    # ᐧ Case 3: New movie
    new_vector = tfidf.transform([movie])
    sim_score = cosine_similarity(new_vector.reshape(1, -1), vectors).flatten()
    sim_indices = sim_score.argsort()[::-1][:11]
    return [new_df['title'].iloc[i].title() for i in sim_indices]

In [36]:
recommend_movie("abcd")

["Doc Holliday'S Revenge",
 'Pocketful Of Miracles',
 'Screwed',
 'Welcome To Collinwood',
 'Under The Same Moon',
 'Housefull',
 "The General'S Daughter",
 'Pontypool',
 'The Thirteenth Floor',
 'The Work And The Glory']